In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import string
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

In [ ]:
DATA_PATH = '/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/data/clean_news_dataset.csv'
MODEL_DIR = '/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/models'
RESULTS_DIR = '/content/drive/MyDrive/NLP_Fake_News_Detection/cit-24-01-0535/results'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('Dataset Shape:', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
print(df.isnull().sum())

In [ ]:
df['clean_text'] = df['clean_text'].fillna('').astype(str)
df['label'] = df['label'].astype(int)

In [ ]:
print('Duplicate Rows:', df.duplicated().sum())
print('Duplicate Content:', df['content'].duplicated().sum())
print('Duplicate Clean Text:', df['clean_text'].duplicated().sum())

In [ ]:
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='label')
plt.xticks([0, 1], ['Fake', 'True'])
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.show()

In [ ]:
X = df['clean_text']
y = df['label']

print('Feature Samples:', X.head(3).tolist())
print('Target Samples:', y.head(3).tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training Samples:', len(X_train))
print('Testing Samples:', len(X_test))
print('Training Class Distribution:')
print(y_train.value_counts())
print('Testing Class Distribution:')
print(y_test.value_counts())

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    lowercase=True,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('TF-IDF Train Shape:', X_train_tfidf.shape)
print('TF-IDF Test Shape:', X_test_tfidf.shape)
print('Vocabulary Size:', len(tfidf.vocabulary_))

In [ ]:
train_vocab = set(tfidf.vocabulary_.keys())
test_only_tokens = set()

for text in X_test:
    for token in text.split():
        if token not in train_vocab:
            test_only_tokens.add(token)

print('TF-IDF Fit on Training Data Only: PASS')
print('Test-only Tokens Not in Training Vocabulary:', len(test_only_tokens))

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)
print('Logistic Regression Training Completed')

In [ ]:
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

print('Predictions Generated:', len(y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=['Fake', 'True'],
    yticklabels=['Fake', 'True']
)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=['Fake', 'True']
))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression ROC Curve')
plt.legend()
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    model,
    X_train_tfidf,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print('Cross-Validation Scores:', cv_scores)
print(f'CV Mean Accuracy: {cv_scores.mean():.4f}')
print(f'CV Standard Deviation: {cv_scores.std():.4f}')

In [ ]:
train_indices = set(X_train.index)
test_indices = set(X_test.index)
print('Train/Test Index Overlap:', len(train_indices.intersection(test_indices)))

In [ ]:
model_path = os.path.join(MODEL_DIR, 'logistic_regression_model.pkl')
vectorizer_path = os.path.join(MODEL_DIR, 'logistic_regression_tfidf_vectorizer.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

with open(vectorizer_path, 'wb') as f:
    pickle.dump(tfidf, f)

print(model_path)
print(vectorizer_path)

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression'],
    'Accuracy': [accuracy],
    'Precision': [precision],
    'Recall': [recall],
    'F1 Score': [f1],
    'ROC-AUC': [roc_auc]
})

results

In [ ]:
results_path = os.path.join(RESULTS_DIR, 'logistic_regression_results.csv')
results.to_csv(results_path, index=False)
print(results_path)

In [ ]:
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

with open(vectorizer_path, 'rb') as f:
    loaded_vectorizer = pickle.load(f)

sample_text = X_test.iloc[0]
sample_vector = loaded_vectorizer.transform([sample_text])
sample_prediction = loaded_model.predict(sample_vector)[0]

print('Actual Label:', y_test.iloc[0])
print('Predicted Label:', sample_prediction)
print('Model Verification Successful')

In [ ]:
print('Final Dataset Shape:', df.shape)
print('Training Samples:', len(X_train))
print('Testing Samples:', len(X_test))
print('Vocabulary Size:', len(tfidf.vocabulary_))
print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')
print(f'ROC-AUC: {roc_auc * 100:.2f}%')
print('Model Path:', model_path)
print('Vectorizer Path:', vectorizer_path)
print('Results Path:', results_path)